# Tema 1 — Explorarea corpusului și primul prompt
În acest notebook vei explora corpusul curățat de comentarii YouTube și vei testa un prim prompt exploratoriu.

Vei testa 10 comentarii și vei reflecta asupra unor probleme precum ambiguitatea, țintele multiple, sarcasmul și confuzia dintre sentiment și poziționarea față de țintă.

## 1. Pregătire
Încărcăm bibliotecile necesare și cheia API pentru Gemini.
Modificați doar celula de configurare a studentului.

In [101]:
%pip install google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [102]:
from pathlib import Path
import os
import json
import random
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [103]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
BASE_URL = "https://api.groq.com/openai/v1"
print("Root project:", ROOT)
print("GROQ key found:", GROQ_API_KEY is not None)

Root project: c:\Users\ASUS\Desktop\ADC 2\INGINERIE AI\echochamber-project-team-2
GROQ key found: True


## 2. Configurare
Modificați  această celulă.
Schimbați `student_id` cu folderul vostru: `student_01`, `student_02`, etc.

In [104]:
student_id = "student_03"
model = "llama-3.1-8b-instant"
temperature = 0.0
corpus_file = ROOT / "data" / "cleaned" / "corpus_youtube_large_clean.jsonl"
output_file = ROOT / "outputs" / f"{student_id}_prompt_outputs.jsonl"

## 3. Încărcăm corpusul curățat
Corpusul este salvat în format JSONL.
JSONL înseamnă: un comentariu pe fiecare linie.

In [105]:
# Citim fiecare linie din fișierul JSONL si o transformăm într-un dataframe pentru explorare
records = []
with corpus_file.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))
# Transformăm lista într-un DataFrame pentru explorare mai ușoară
df = pd.DataFrame(records)
df.head()

,id,source_platform,source_channel,text,text_raw,bubble_label,bubble_self_identified,topic,rhetoric_type,video_id,video_title,video_date,comment_date,likes,lang,collected_at
0,yt_5rHoTX3U_3Q_UgxaV5so7vyeXpyy8up4AaABAg,youtube,georgesimionoficial,Felicitării George Simion Președintele României!🇷🇴😇⛑️🇷🇴❤️🛐❤️✝️✝️✝️❤️🕯️💐🇷🇴🙌💯🙌.,Felicitării George Simion Președintele României!🇷🇴😇⛑️🇷🇴❤️🛐❤️✝️✝️✝️❤️🕯️💐🇷🇴🙌💯🙌.,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-23,1,ro,2026-03-22
1,yt_5rHoTX3U_3Q_UgwJYiRLMbLfl2AipVR4AaABAg,youtube,georgesimionoficial,Asa trebuie să fiți printre oameni nu sa se doarmă in parlament succes domnule Simion nu am greșit cind v-am votat,Asa trebuie să fiți printre oameni nu sa se doarmă in parlament succes domnule Simion nu am greșit cind v-am votat,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-02,5,ro,2026-03-22
2,yt_5rHoTX3U_3Q_UgzXqOk_SypZcQS-JcF4AaABAg,youtube,georgesimionoficial,"Eu am votat cu George Simion din primul tur ptr că am vrut să avem președintele României un tânăr Roman,un patriot și un luptător. Încă nu mi-am pierdut speranța, sper să ne facem bine.","Eu am votat cu George Simion din primul tur ptr că am vrut să avem președintele României un tânăr Roman,un patriot și un luptător. Încă nu mi-am pierdut speranța, sper să ne facem bine.",None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,30,ro,2026-03-22
3,yt_5rHoTX3U_3Q_UgzpKghDX0_l-Gc3P4V4AaABAg,youtube,georgesimionoficial,"Si de trebuie deposite de combustibil degeaba avem tancuri,blindate etc daca stau parcate ,cauta sursele de petrol care le are romania si daca nu avem Sau nu suficient cumparam CAT de mult putem deposita","Si de trebuie deposite de combustibil degeaba avem tancuri,blindate etc daca stau parcate ,cauta sursele de petrol care le are romania si daca nu avem Sau nu suficient cumparam CAT de mult putem deposita",None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-16,3,ro,2026-03-22
4,yt_5rHoTX3U_3Q_UgwqOTRSHNPj9cuGwNt4AaABAg,youtube,georgesimionoficial,"Nu te descuraja că dobitoci și proști vor fi peste tot nu poti sa-i mulțumești pe toți, cu D-ZEU înainte, UNITATE LIBERTATE, PACE, DEMOCRAȚIE. FORȚA AUR, GEORGE SIMION. JOS GUVERNUL ROMÂNIEI. DEMISIA NICUȘOR Dan BOLOJAN PREDOIU VEXLER FRITZ LUDOVIC TERHES KELEMEN HUNOR BARNA CÎȚU..... șamd. CĂLIN GEORGESCU PREȘEDINTELE ROMÂNIEI.","Nu te descuraja că dobitoci și proști vor fi peste tot nu poti sa-i mulțumești pe toți, cu D-ZEU înainte, UNITATE LIBERTATE, PACE, DEMOCRAȚIE. FORȚA AUR, GEORGE SIMION. JOS GUVERNUL ROMÂNIEI. DEMISIA NICUȘOR Dan BOLOJAN PREDOIU VEXLER FRITZ LUDOVIC TERHES KELEMEN HUNOR BARNA CÎȚU..... șamd.\nCĂLIN GEORGESCU PREȘEDINTELE ROMÂNIEI.",None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,9,ro,2026-03-22


In [106]:
print("Number of comments:", len(df))
print("Columns:", list(df.columns))

Number of comments: 30451
Columns: ['id', 'source_platform', 'source_channel', 'text', 'text_raw', 'bubble_label', 'bubble_self_identified', 'topic', 'rhetoric_type', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'lang', 'collected_at']


## 4. Explorare rapidă a corpusului
Ne uităm la canalele principale și la câteva exemple de comentarii.
Această etapă ne ajută să înțelegem ce tip de date avem înainte să folosim modelul.

In [107]:
# cele mai frecvente 15 canale sursă din dataset
df["source_channel"].value_counts().head(15) # completează pentru a vedea cele mai frecvente 15 canale sursă din dataset

source_channel
RecorderRomania                   12177
turcescu111                        5019
georgesimionoficial                3669
CălinGeorgescu-CanalulOficial      3460
@CălinGeorgescu-CanalulOficial     2557
TuDecizi-s3g                        647
StareaNatiei                        623
AltcevacuAdrianArtene               363
roxindaniel                         305
otvdirect                           304
digi24hd56                          265
euronewsro                          238
DianaSosoacaOfficial                227
AdevaruriSecrete                    180
g4media479                          158
Name: count, dtype: int64

In [108]:
# aruncă o privire asupra unor comentarii random din dataset
df[["source_channel", "video_title", "text"]].sample(5, random_state=42)

,source_channel,video_title,text
23002,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ - 25.08.2025 ),"Multă sănătate dl. Președinte Călin Georgescu. Noi mergem până la capăt și vă dorim să fiți acceptat măcar președinte de scară de bloc. Și să nu fugiți în Dubai, rămâneți alături de noi. 🇷🇴"
5815,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu când ne e greu? ( 2026.01.15 ),"Un discurs care trebuia sa vina de la Cotroceni! Multumim, d_le Calin Georgescu!"
11191,RecorderRomania,"Primarul Negoiță a construit șosele peste magistralele de gaz. „La o explozie, nu mai rămâne nimic!”",Autoritatile abilitate sa intervina!!! De aceea sunt platiti de noi! Multumim recorder!!❤
11316,RecorderRomania,"Primarul Negoiță a construit șosele peste magistralele de gaz. „La o explozie, nu mai rămâne nimic!”",In acest moment mai putem spune doar Dumnezeu sa ii apere pe cei care stau acolo sau trec pe acolo cu masina 😩
12505,RecorderRomania,DOCUMENTAR RECORDER. Singuri,"E dureros.. e crunt.. simt vinovatie si recunostinta in acelasi timp pentru ca mie nu mi-a lipsit dragostea mamei si caldura familiei. Imi pare atat de rau ca nu toti au avut parte de asta ....cel mai tare doare ca ei traiesc cu traumele astea toata viata. Trecem pe langa oameni pe strada si in ignoranta noastra poate judecam cu superficialitate un gest, un cuvant urat pe care il auzim, dar nici prin minte nu ne trece ca omul acela altceva nu a vazut, nu a auzit.. sunt ingrozita de acest sistem bolnav care nu face nimic bun.. ""protectia copilului"".... ce-o mai fi si aia...Multumim, Recorder pentru ca ne puneti fata in fata cu o realitate de care nu suntem constienti!"


## 5. Alegem 10 comentarii pentru testarea promptului
Folosim 10 comentarii curate.
Puteți păstra eșantionarea aleatorie sau puteți selecta manual comentarii mai interesante.

In [109]:
sample_df = df.sample(10, random_state=42).copy()
sample_df[["source_channel", "text"]]

,source_channel,text
23002,CălinGeorgescu-CanalulOficial,"Multă sănătate dl. Președinte Călin Georgescu. Noi mergem până la capăt și vă dorim să fiți acceptat măcar președinte de scară de bloc. Și să nu fugiți în Dubai, rămâneți alături de noi. 🇷🇴"
5815,@CălinGeorgescu-CanalulOficial,"Un discurs care trebuia sa vina de la Cotroceni! Multumim, d_le Calin Georgescu!"
11191,RecorderRomania,Autoritatile abilitate sa intervina!!! De aceea sunt platiti de noi! Multumim recorder!!❤
11316,RecorderRomania,In acest moment mai putem spune doar Dumnezeu sa ii apere pe cei care stau acolo sau trec pe acolo cu masina 😩
12505,RecorderRomania,"E dureros.. e crunt.. simt vinovatie si recunostinta in acelasi timp pentru ca mie nu mi-a lipsit dragostea mamei si caldura familiei. Imi pare atat de rau ca nu toti au avut parte de asta ....cel mai tare doare ca ei traiesc cu traumele astea toata viata. Trecem pe langa oameni pe strada si in ignoranta noastra poate judecam cu superficialitate un gest, un cuvant urat pe care il auzim, dar nici prin minte nu ne trece ca omul acela altceva nu a vazut, nu a auzit.. sunt ingrozita de acest sistem bolnav care nu face nimic bun.. ""protectia copilului"".... ce-o mai fi si aia...Multumim, Recorder pentru ca ne puneti fata in fata cu o realitate de care nu suntem constienti!"
9644,RecorderRomania,Cite dosare ați judecat și nu ați recuperat nici un prejudiciu
23843,turcescu111,"Totul duce către: Noua Ordine Mondială, pentru a subjuga întreaga omenire..... Ei, și prin Agenda 2030, carmuieste într-acolo..... Și da, vor să ne omoare 90% din populatie.... Da, așa este o 'crimă' mondială, al treilea război mondial."
11605,RecorderRomania,"Un hot corupt arogant si nesimtit, caruia nimeni nu-i face nimic! 😡 Este ingrozitor!"
15486,RecorderRomania,"4:30 și încă 1% rămas pentru Crin Alcoolescu, să aibă și el acolo ceva săracul"
7767,RecorderRomania,Vă mai dau niște firme din Galați care au alți administratori dar cu foști patroni


Optional , poti alege sa folosesti  alta metoda de esantionare sau sa filtrezi dupa anumite canale sursa sau alte criterii. Important e sa ai un set de date mic pe care sa testezi promptul inainte de a-l rula pe intregul dataset.

## 6. Primul prompt exploratoriu
Completăm un prompt simplu pentru analizarea comentariilor politice.
Promptul trebuie să ceară:
- ținta comentariului;
- poziționarea față de țintă;
- tonul;
- tema;
- problema de interpretare;
- o justificare scurtă.
Important: tonul sau sentimentul general nu este același lucru cu poziționarea față de țintă.

In [126]:
USER_PROMPT_TEMPLATE = """
Analizează comentariul politic românesc.

Returnează DOAR JSON valid cu exact aceste chei:
target, stance, sentiment, tone, topic, interpretation_problem

REGULĂ PRINCIPALĂ:
stance trebuie să fie "neutru" dacă textul NU susține sau NU critică explicit suveraniștii.

====================
target
====================

Valori posibile:
- suveranisti
- anti_suveranisti
- politicieni
- guvern
- sistem
- jurnalisti
- ue
- nato
- romania
- electorat
- unknown

Reguli target:
- George Simion, Călin Georgescu, AUR -> suveranisti
- Recorder, jurnaliști, reporteri, documentare -> jurnalisti
- guvern, stat, instituții -> guvern
- sistemul, statul paralel -> sistem
- UE, Bruxelles -> ue
- NATO -> nato
- România ca țară/națiune -> romania
- poporul, românii, votanții -> electorat
- dacă nu e clar -> unknown

====================
stance
====================

Valori posibile:
- pro_suveranist
- anti_suveranist
- neutru

Reguli stance:

Alege pro_suveranist DOAR dacă textul laudă sau susține explicit:
- George Simion
- Călin Georgescu
- AUR
- suveranismul
- discurs anti-UE / anti-NATO / anti-globalist

Alege anti_suveranist DOAR dacă textul critică explicit:
- George Simion
- Călin Georgescu
- AUR
- suveranismul
- naționalismul politic

Alege neutru pentru orice altceva:
- critică guvernul
- critică statul
- critică sistemul
- critică instituțiile
- critică societatea
- vorbește despre popor
- vorbește despre corupție
- vorbește despre justiție
- vorbește despre educație
- laudă Recorder
- patriotism generic
- comentariu emoțional fără referință clară la suveraniști

IMPORTANT:
- NU folosi anti_suveranist doar pentru că sentimentul este negativ.
- NU folosi anti_suveranist doar pentru că targetul este guvern, sistem sau electorat.
- NU folosi pro_suveranist doar pentru că apare România sau poporul.
- Dacă ai dubii, stance = neutru.

Exemple:
Comentariu: "Unde sunt banii poporului? Instituțiile fură."
Output: {{"target":"guvern","stance":"neutru","sentiment":"negativ","tone":"frustrat","topic":"coruptie","interpretation_problem":"none"}}

Comentariu: "Călin Georgescu este singurul președinte legitim."
Output: {{"target":"suveranisti","stance":"pro_suveranist","sentiment":"pozitiv","tone":"admirativ","topic":"leadership_politic","interpretation_problem":"none"}}

Comentariu: "Nu votez Simion."
Output: {{"target":"suveranisti","stance":"anti_suveranist","sentiment":"negativ","tone":"neutru","topic":"alegeri","interpretation_problem":"none"}}

Comentariu: "Respect Recorder pentru documentar."
Output: {{"target":"jurnalisti","stance":"neutru","sentiment":"pozitiv","tone":"admirativ","topic":"jurnalism","interpretation_problem":"none"}}

====================
sentiment
====================

Valori posibile:
- pozitiv
- negativ
- neutru
- mixt

====================
tone
====================

Valori posibile:
- agresiv
- ironic
- sarcastic
- conspirativ
- patriotic
- frustrat
- admirativ
- emotional
- rational
- neutru

====================
topic
====================

Valori posibile:
- alegeri
- justitie
- propaganda
- manipulare_media
- ue
- nato
- coruptie
- economie
- identitate_nationala
- leadership_politic
- jurnalism
- educatie
- unknown

====================
interpretation_problem
====================

Valori posibile:
- sarcasm
- ironie
- ambiguitate
- limbaj_vulgar
- typo
- none

IMPORTANT:
- sentiment și stance NU sunt același lucru.
- un comentariu poate avea sentiment negativ și stance neutru.
- criticismul, furia sau frustrarea NU implică automat anti_suveranism sau pro_suveranism.
- stance trebuie determinat DOAR din poziționarea explicită față de:
  George Simion,
  Călin Georgescu,
  AUR,
  suveranism,
  UE,
  NATO,
  globalism.

Exemple:
- "Țara este coruptă" -> sentiment negativ, stance neutru
- "Guvernul fură" -> sentiment negativ, stance neutru
- "UE distruge România" -> stance pro_suveranist
- "Nu votez Simion" -> stance anti_suveranist

Comentariu:
<<< {comment_text} >>>
"""

## 7. Conectarea la model
Folosim Gemini prin endpoint compatibil cu OpenAI.
Modelul și temperatura au fost setate mai sus.

In [115]:
from openai import OpenAI
client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

In [127]:
def annotate_comment(comment_text):
    prompt = USER_PROMPT_TEMPLATE.format(comment_text=comment_text)
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

## 8. Rulăm promptul pe 10 comentarii
Trimitem fiecare comentariu selectat la model și salvăm răspunsurile.

In [128]:
n_comments = 10
sample_for_prompt = df.sample(n_comments)

outputs = []

for _, row in sample_for_prompt.iterrows():
    outputs.append({
        "source_channel": row.get("source_channel", ""),
        "video_title": row.get("video_title", ""),
        "comment_text": row["text"],
        "model_output": annotate_comment(row["text"])
    })

results_df = pd.DataFrame(outputs)

results_df

,source_channel,video_title,comment_text,model_output
0,turcescu111,Harpalete- Sângerete și transfuzia din lumea lui,"Totul pleacă de la lipsa de educație a noastră. Atunci când vezi peste tot lehamite, ajungi să crezi că așa trebuie să fie. Când peste tot se ocupă posturi doar cu pile și șpagă...","{\n ""target"": ""sistem"",\n ""stance"": ""neutru"",\n ""sentiment"": ""negativ"",\n ""tone"": ""frustrat"",\n ""topic"": ""educatie"",\n ""interpretation_problem"": ""none""\n}"
1,turcescu111,Psihiatria salvează România!,"Concubina locuiește ilegal la Cotroceni, uitați-vă bine la Iran ca Mucifer ne va baga an război civil antre frații scârbă este răzbunătoare","{\n ""target"": ""guvern"",\n ""stance"": ""neutru"",\n ""sentiment"": ""negativ"",\n ""tone"": ""conspirativ"",\n ""topic"": ""coruptie"",\n ""interpretation_problem"": ""ambiguitate""\n}"
2,otvdirect,Profesor român executat în Franța! Șoc total,Prostanacu ' ăsta din Franta stie el că Germania Franta Spania ltalia ar veni in Romania daca ar fi atacata . Toti astia o maninca pe Romania de toate bunurile . Ce a construit SUA Deveselu Kogalniceanu si altele sunt ale SUA . ...nato nu erau si europenii . ?,"```json\n{\n ""target"": ""ue"",\n ""stance"": ""pro_suveranist"",\n ""sentiment"": ""negativ"",\n ""tone"": ""agresiv"",\n ""topic"": ""ue"",\n ""interpretation_problem"": ""none""\n}\n```"
3,georgesimionoficial,"Romania are un președinte, numele lui este CALIN GEORGESCU",DUMNEZEU SA ADUCA IZBAVIRE SI DREPTATE. VOTEZ CU MARE DRAG CALIN GEOGESCU PRESEDINTE ❤❤❤❤,"```json\n{\n ""target"": ""suveranisti"",\n ""stance"": ""pro_suveranist"",\n ""sentiment"": ""pozitiv"",\n ""tone"": ""patriotic"",\n ""topic"": ""leadership_politic"",\n ""interpretation_problem"": ""none""\n}\n```"
4,RecorderRomania,Reportaj Recorder în noaptea alegerilor: Un președinte peste mai multe Românii,"Mi-e mila de oamenii care ai fost mintiti de Realitatea TV,sper ca statul sa vada de ce oamenii acestia au fost dezinformati!","{\n ""target"": ""guvern"",\n ""stance"": ""neutru"",\n ""sentiment"": ""negativ"",\n ""tone"": ""emotional"",\n ""topic"": ""propaganda"",\n ""interpretation_problem"": ""none""\n}"
5,georgesimionoficial,În 30 de minute începe evenimentul de la Roma: câteva clarificări!,Se poate repara ce sa distrus dar în mulți ani de zile (10).,"```json\n{\n ""target"": ""sistem"",\n ""stance"": ""neutru"",\n ""sentiment"": ""negativ"",\n ""tone"": ""frustrat"",\n ""topic"": ""economie"",\n ""interpretation_problem"": ""none""\n}\n```"
6,georgesimionoficial,"Respect pentru muncă! În dialog cu poporul român, zi de zi!",George să nu uiți niciodată orice zi sa o incepi cu Dumnezeu și noi te susținem in rugăciune dar e nevoie ca și tu zilnic sa ai o relație cu Dumnezeu prin rugăciune Dumnezeu să te ajute in tot ce faci și să-ți dea înțelepciune in fata hienilor sa fii cel mai înțelept răspunsuri pe măsură 🙏🙏🙏,"```json\n{\n ""target"": ""suveranisti"",\n ""stance"": ""pro_suveranist"",\n ""sentiment"": ""pozitiv"",\n ""tone"": ""patriotic"",\n ""topic"": ""leadership_politic"",\n ""interpretation_problem"": ""none""\n}\n```"
7,otvdirect,"SORIN CONSTANTINESCU, STEFAN JICOL||DAN DIACONESCU DIRECT",nu poti scoate propaganda dintr-un cap neted... nea Pacanel e nivelat de propaganda... patineaza musca pe creierul lui. zero circumvolutiuni.,"{\n ""target"": ""jurnalisti"",\n ""stance"": ""neutru"",\n ""sentiment"": ""negativ"",\n ""tone"": ""agresiv"",\n ""topic"": ""propaganda"",\n ""interpretation_problem"": ""sarcasm""\n}"
8,StareaNatiei,"De ce ajungem să dăm vina pe oricine, mai puțin pe adevărații responsabili? | Starea Nației 19.03.26",Aici doar pentru un comentariu ca emisiunea am vazut-o la tv :),"{\n ""target"": ""jurnalisti"",\n ""stance"": ""neutru"",\n ""sentiment"": ""pozitiv"",\n ""tone"": ""emotional"",\n ""topic"": ""jurnalism"",\n ""interpretation_problem"": ""none""\n}"
9,RecorderRomania,EXPLICATIV RECORDER. Cum s-au repliat liderii sistemului după documentarul „Justiție capturată”,"Domn

# 9. Verificam rezultatele

In [129]:
results_df.model_output[0]

'{\n  "target": "sistem",\n  "stance": "neutru",\n  "sentiment": "negativ",\n  "tone": "frustrat",\n  "topic": "educatie",\n  "interpretation_problem": "none"\n}'

In [130]:
# funcție pentru a curăța și parsa output-ul modelului, care poate conține JSON în diferite formate (text simplu sau bloc ```json)

def parse_model_output(text):
    # Modelul poate întoarce JSON ca text simplu sau în bloc ```json
    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()
    
    return json.loads(text)

In [131]:
parsed_outputs = []

for _, row in results_df.iterrows():
    parsed = parse_model_output(row["model_output"])
    
    parsed_outputs.append({
        "source_channel": row["source_channel"],
        "video_title": row["video_title"],
        "comment_text": row["comment_text"],
        "target": parsed.get("target", ""),
        "stance": parsed.get("stance", ""),
        "sentiment": parsed.get("sentiment", ""),
        "tone": parsed.get("tone", ""),
        "topic": parsed.get("topic", ""),
        "interpretation_problem": parsed.get("interpretation_problem", ""),
        "reason": parsed.get("reason", "")
    })

parsed_df = pd.DataFrame(parsed_outputs)
parsed_df

,source_channel,video_title,comment_text,target,stance,sentiment,tone,topic,interpretation_problem,reason
0,turcescu111,Harpalete- Sângerete și transfuzia din lumea lui,"Totul pleacă de la lipsa de educație a noastră. Atunci când vezi peste tot lehamite, ajungi să crezi că așa trebuie să fie. Când peste tot se ocupă posturi doar cu pile și șpagă...",sistem,neutru,negativ,frustrat,educatie,none,
1,turcescu111,Psihiatria salvează România!,"Concubina locuiește ilegal la Cotroceni, uitați-vă bine la Iran ca Mucifer ne va baga an război civil antre frații scârbă este răzbunătoare",guvern,neutru,negativ,conspirativ,coruptie,ambiguitate,
2,otvdirect,Profesor român executat în Franța! Șoc total,Prostanacu ' ăsta din Franta stie el că Germania Franta Spania ltalia ar veni in Romania daca ar fi atacata . Toti astia o maninca pe Romania de toate bunurile . Ce a construit SUA Deveselu Kogalniceanu si altele sunt ale SUA . ...nato nu erau si europenii . ?,ue,pro_suveranist,negativ,agresiv,ue,none,
3,georgesimionoficial,"Romania are un președinte, numele lui este CALIN GEORGESCU",DUMNEZEU SA ADUCA IZBAVIRE SI DREPTATE. VOTEZ CU MARE DRAG CALIN GEOGESCU PRESEDINTE ❤❤❤❤,suveranisti,pro_suveranist,pozitiv,patriotic,leadership_politic,none,
4,RecorderRomania,Reportaj Recorder în noaptea alegerilor: Un președinte peste mai multe Românii,"Mi-e mila de oamenii care ai fost mintiti de Realitatea TV,sper ca statul sa vada de ce oamenii acestia au fost dezinformati!",guvern,neutru,negativ,emotional,propaganda,none,
5,georgesimionoficial,În 30 de minute începe evenimentul de la Roma: câteva clarificări!,Se poate repara ce sa distrus dar în mulți ani de zile (10).,sistem,neutru,negativ,frustrat,economie,none,
6,georgesimionoficial,"Respect pentru muncă! În dialog cu poporul român, zi de zi!",George să nu uiți niciodată orice zi sa o incepi cu Dumnezeu și noi te susținem in rugăciune dar e nevoie ca și tu zilnic sa ai o relație cu Dumnezeu prin rugăciune Dumnezeu să te ajute in tot ce faci și să-ți dea înțelepciune in fata hienilor sa fii cel mai înțelept răspunsuri pe măsură 🙏🙏🙏,suveranisti,pro_suveranist,pozitiv,patriotic,leadership_politic,none,
7,otvdirect,"SORIN CONSTANTINESCU, STEFAN JICOL||DAN DIACONESCU DIRECT",nu poti scoate propaganda dintr-un cap neted... nea Pacanel e nivelat de propaganda... patineaza musca pe creierul lui. zero circumvolutiuni.,jurnalisti,neutru,negativ,agresiv,propaganda,sarcasm,
8,StareaNatiei,"De ce ajungem să dăm vina pe oricine, mai puțin pe adevărații responsabili? | Starea Nației 19.03.26",Aici doar pentru un comentariu ca emisiunea am vazut-o la tv :),jurnalisti,neutru,pozitiv,emotional,jurnalism,none,
9,RecorderRomania,EXPLICATIV RECORDER. Cum s-au repliat liderii sistemului după documentarul „Justiție capturată”,"Domnule Nicusor Oameni nu sunt proști, văd adevărul tu nu-l vezi ?. bravo Recorder ! Adevărul mai presus de corupție",jurnalisti,neutru,pozitiv,admirativ,jurnalism,none,


In [132]:
len(parsed_df)

10

# 10 Salvarea csv si inspectarea rezulatelor
- salvati ca csv 
- deschideti csv si verificati rezultatele 
- raspundeti la urmatoarele intrebare: promptul separă corect sentimentul general de poziționarea față de target? 

In [133]:
parsed_df.to_csv(r"C:\Users\ASUS\Desktop\ADC 2\INGINERIE AI\echochamber-project-team-2\notebooks\student_03\tema1_stud03_output.csv",
                index=False, encoding="utf-8-sig")

In [134]:
csv_output = pd.read_csv(r"C:\Users\ASUS\Desktop\ADC 2\INGINERIE AI\echochamber-project-team-2\notebooks\student_03\tema1_stud03_output.csv")

In [135]:
csv_output.head(10)

,source_channel,video_title,comment_text,target,stance,sentiment,tone,topic,interpretation_problem,reason
0,turcescu111,Harpalete- Sângerete și transfuzia din lumea lui,"Totul pleacă de la lipsa de educație a noastră. Atunci când vezi peste tot lehamite, ajungi să crezi că așa trebuie să fie. Când peste tot se ocupă posturi doar cu pile și șpagă...",sistem,neutru,negativ,frustrat,educatie,none,NaN
1,turcescu111,Psihiatria salvează România!,"Concubina locuiește ilegal la Cotroceni, uitați-vă bine la Iran ca Mucifer ne va baga an război civil antre frații scârbă este răzbunătoare",guvern,neutru,negativ,conspirativ,coruptie,ambiguitate,NaN
2,otvdirect,Profesor român executat în Franța! Șoc total,Prostanacu ' ăsta din Franta stie el că Germania Franta Spania ltalia ar veni in Romania daca ar fi atacata . Toti astia o maninca pe Romania de toate bunurile . Ce a construit SUA Deveselu Kogalniceanu si altele sunt ale SUA . ...nato nu erau si europenii . ?,ue,pro_suveranist,negativ,agresiv,ue,none,NaN
3,georgesimionoficial,"Romania are un președinte, numele lui este CALIN GEORGESCU",DUMNEZEU SA ADUCA IZBAVIRE SI DREPTATE. VOTEZ CU MARE DRAG CALIN GEOGESCU PRESEDINTE ❤❤❤❤,suveranisti,pro_suveranist,pozitiv,patriotic,leadership_politic,none,NaN
4,RecorderRomania,Reportaj Recorder în noaptea alegerilor: Un președinte peste mai multe Românii,"Mi-e mila de oamenii care ai fost mintiti de Realitatea TV,sper ca statul sa vada de ce oamenii acestia au fost dezinformati!",guvern,neutru,negativ,emotional,propaganda,none,NaN
5,georgesimionoficial,În 30 de minute începe evenimentul de la Roma: câteva clarificări!,Se poate repara ce sa distrus dar în mulți ani de zile (10).,sistem,neutru,negativ,frustrat,economie,none,NaN
6,georgesimionoficial,"Respect pentru muncă! În dialog cu poporul român, zi de zi!",George să nu uiți niciodată orice zi sa o incepi cu Dumnezeu și noi te susținem in rugăciune dar e nevoie ca și tu zilnic sa ai o relație cu Dumnezeu prin rugăciune Dumnezeu să te ajute in tot ce faci și să-ți dea înțelepciune in fata hienilor sa fii cel mai înțelept răspunsuri pe măsură 🙏🙏🙏,suveranisti,pro_suveranist,pozitiv,patriotic,leadership_politic,none,NaN
7,otvdirect,"SORIN CONSTANTINESCU, STEFAN JICOL||DAN DIACONESCU DIRECT",nu poti scoate propaganda dintr-un cap neted... nea Pacanel e nivelat de propaganda... patineaza musca pe creierul lui. zero circumvolutiuni.,jurnalisti,neutru,negativ,agresiv,propaganda,sarcasm,NaN
8,StareaNatiei,"De ce ajungem să dăm vina pe oricine, mai puțin pe adevărații responsabili? | Starea Nației 19.03.26",Aici doar pentru un comentariu ca emisiunea am vazut-o la tv :),jurnalisti,neutru,pozitiv,emotional,jurnalism,none,NaN
9,RecorderRomania,EXPLICATIV RECORDER. Cum s-au repliat liderii sistemului după documentarul „Justiție capturată”,"Domnule Nicusor Oameni nu sunt proști, văd adevărul tu nu-l vezi ?. bravo Recorder ! Adevărul mai presus de corupție",jurnalisti,neutru,pozitiv,admirativ,jurnalism,none,NaN


In [136]:
pd.set_option("display.max_colwidth", None)

csv_output[["comment_text", "target", "stance", "sentiment"]].head(10)

,comment_text,target,stance,sentiment
0,"Totul pleacă de la lipsa de educație a noastră. Atunci când vezi peste tot lehamite, ajungi să crezi că așa trebuie să fie. Când peste tot se ocupă posturi doar cu pile și șpagă...",sistem,neutru,negativ
1,"Concubina locuiește ilegal la Cotroceni, uitați-vă bine la Iran ca Mucifer ne va baga an război civil antre frații scârbă este răzbunătoare",guvern,neutru,negativ
2,Prostanacu ' ăsta din Franta stie el că Germania Franta Spania ltalia ar veni in Romania daca ar fi atacata . Toti astia o maninca pe Romania de toate bunurile . Ce a construit SUA Deveselu Kogalniceanu si altele sunt ale SUA . ...nato nu erau si europenii . ?,ue,pro_suveranist,negativ
3,DUMNEZEU SA ADUCA IZBAVIRE SI DREPTATE. VOTEZ CU MARE DRAG CALIN GEOGESCU PRESEDINTE ❤❤❤❤,suveranisti,pro_suveranist,pozitiv
4,"Mi-e mila de oamenii care ai fost mintiti de Realitatea TV,sper ca statul sa vada de ce oamenii acestia au fost dezinformati!",guvern,neutru,negativ
5,Se poate repara ce sa distrus dar în mulți ani de zile (10).,sistem,neutru,negativ
6,George să nu uiți niciodată orice zi sa o incepi cu Dumnezeu și noi te susținem in rugăciune dar e nevoie ca și tu zilnic sa ai o relație cu Dumnezeu prin rugăciune Dumnezeu să te ajute in tot ce faci și să-ți dea înțelepciune in fata hienilor sa fii cel mai înțelept răspunsuri pe măsură 🙏🙏🙏,suveranisti,pro_suveranist,pozitiv
7,nu poti scoate propaganda dintr-un cap neted... nea Pacanel e nivelat de propaganda... patineaza musca pe creierul lui. zero circumvolutiuni.,jurnalisti,neutru,negativ
8,Aici doar pentru un comentariu ca emisiunea am vazut-o la tv :),jurnalisti,neutru,pozitiv
9,"Domnule Nicusor Oameni nu sunt proști, văd adevărul tu nu-l vezi ?. bravo Recorder ! Adevărul mai presus de corupție",jurnalisti,neutru,pozitiv


### Modelul separa in general bine pozitionarea fata de target si sentimentul general al comentariului. Identifica cu succes naratiunile pro-suveraniste si nu identifica stance-uri pro sau contra suveraniste daca nu e cazul. Probleme la identificarea target-ului corect, de exemplu comentariul numarul 7. 

###